# Notebook 1: explore the data

Bank customer churn, same problem as in the slides.

Load the CSV, check dtypes and missing values, plot a few things. Do this before training anything.

Next: `02_train_track_register.ipynb`.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px

# Works whether you started Jupyter from /notebooks or the repo root.
DEMO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = DEMO_ROOT / "data" / "bank_customers.csv"

## Load and preview

`data/bank_customers.csv` has one row per customer. `churn` is the label: 1 means they left.

In [2]:
df = pd.read_csv(DATA_PATH)
df.head()

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## Schema and types

Row count, dtypes, nulls. Fix anything weird here before you train.

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customer_id       10000 non-null  int64  
 1   credit_score      10000 non-null  int64  
 2   country           10000 non-null  object 
 3   gender            10000 non-null  object 
 4   age               10000 non-null  int64  
 5   tenure            10000 non-null  int64  
 6   balance           10000 non-null  float64
 7   products_number   10000 non-null  int64  
 8   credit_card       10000 non-null  int64  
 9   active_member     10000 non-null  int64  
 10  estimated_salary  10000 non-null  float64
 11  churn             10000 non-null  int64  
dtypes: float64(2), int64(8), object(2)
memory usage: 937.6+ KB


## Summary stats

Ranges for numerics, cardinality for categoricals. Good place to spot outliers.

In [4]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customer_id,10000.0,NaN,NaN,NaN,15690940.5694,71936.186123,15565701.0,15628528.25,15690738.0,15753233.75,15815690.0
credit_score,10000.0,NaN,NaN,NaN,650.5288,96.653299,350.0,584.0,652.0,718.0,850.0
country,10000,3,France,5014,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,10000,2,Male,5457,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,10000.0,NaN,NaN,NaN,38.9218,10.487806,18.0,32.0,37.0,44.0,92.0
tenure,10000.0,NaN,NaN,NaN,5.0128,2.892174,0.0,3.0,5.0,7.0,10.0
balance,10000.0,NaN,NaN,NaN,76485.889288,62397.405202,0.0,0.0,97198.54,127644.24,250898.09
products_number,10000.0,NaN,NaN,NaN,1.5302,0.581654,1.0,1.0,1.0,2.0,4.0
credit_card,10000.0,NaN,NaN,NaN,0.7055,0.45584,0.0,0.0,1.0,1.0,1.0
active_member,10000.0,NaN,NaN,NaN,0.5151,0.499797,0.0,0.0,1.0,1.0,1.0


## Target distribution

Churn is imbalanced (more stayers than leavers). Accuracy alone will look fine and tell you almost nothing.

In [5]:
counts = df["churn"].value_counts().sort_index()
px.bar(
    x=["No churn (0)", "Churn (1)"],
    y=counts.values,
    title="Churn class distribution",
    color=["No churn (0)", "Churn (1)"],
    color_discrete_sequence=["#2ecc71", "#e74c3c"],
).update_layout(showlegend=False).show()

## Numeric features vs churn

Violin plots for `age`, `balance`, `credit_score`. Do churners look different?

In [6]:
for col in ["age", "balance", "credit_score"]:
    px.violin(
        df,
        x="churn",
        y=col,
        color="churn",
        box=True,
        points=False,
        title=f"{col.replace('_', ' ').title()} by churn",
        color_discrete_map={0: "#2ecc71", 1: "#e74c3c"},
    ).show()

## Categorical features vs churn

Bar charts for `products_number`, `active_member`, `country`, `gender`. These get one-hot encoded in notebook 2.

In [7]:
for col in ["products_number", "active_member", "country", "gender"]:
    grouped = df.groupby([col, "churn"]).size().reset_index(name="count")
    px.bar(
        grouped,
        x=col,
        y="count",
        color="churn",
        barmode="group",
        title=f"{col.replace('_', ' ').title()} vs churn",
        color_discrete_map={0: "#2ecc71", 1: "#e74c3c"},
    ).show()

## Correlations

Heatmap of numeric columns. We keep everything for the demo, but worth a glance if two features are basically the same signal.

In [8]:
corr = df.select_dtypes(include="number").corr()
px.imshow(
    corr,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu_r",
    title="Feature correlation heatmap",
).show()

## Summary

Two questions worth answering before you train anything:

**Is the target imbalanced?**  
Yes. Most customers stay; churn is the minority class. That's why accuracy is a bad headline metric later.

**Which features actually separate churners from everyone else?**  
The plots above aren't subtle. Age and balance move together: churners skew older and carry higher balances. Product count matters too, but not in a linear way. Membership status and gender show up clearly in the bar charts.

**Patterns worth remembering:**

- Older customers with higher balances churn more. The violin plots for `age` and `balance` both shift right for churn=1.
- Customers with 3+ products churn disproportionately. One or two products looks relatively safe; three or four is where it gets ugly.
- Inactive members (`active_member=0`) churn far more often than active ones. Probably the strongest single signal in the categoricals.
- Female customers show higher churn in this dataset. Don't over-read it without checking sample sizes per group, but the gap is visible.

None of this is modeling yet. It's just deciding what the problem looks like. Notebook 2 picks up from here.